# Session 5 · Spectral analysis: power spectra & band power

On Friday and Saturday you saw the night as a **spectrogram**. Today we take it apart: the **power spectrum** (PSD) of an epoch, of a stage, of a night, and the **band power** numbers that summarise it. These numbers become the EEG features of the cohort table, and on Wednesday the input of the age prediction.

You will
1. compute the spectrum of **one epoch**, and see how the window length trades **frequency resolution** against **noise**
2. average the spectra **per stage**, on the clean epochs only
3. check what **artefact rejection** changes in those spectra
4. compute **band power**, absolute and relative, for every epoch, and save the table
5. follow **slow-wave activity** and other bands across the night
6. compare a **young and an old** subject, including the **1/f slope**: with a straight line, and with **specparam**.

**How to work**
- Mostly **run, change, look**: cells marked 👉 have a parameter to change and rerun. The `# TODO` cells are short.
- Night: **SC4002** again, so you can compare with what you know.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa
from scipy.integrate import simpson
from scipy.signal import welch
from specparam import SpectralModel

sys.path.insert(0, str(Path("..").resolve()))  # so Python finds sleep_utils.py in the repo folder
from sleep_utils import SUBJECTS_CSV, epoch_onsets, load_epochs

mne.set_log_level("WARNING")

RESULTS_DIR = Path("..") / "results"
NIGHT_DIR = RESULTS_DIR / "SC4002"
subjects = pd.read_csv(SUBJECTS_CSV)

STAGES = ["WAKE", "N1", "N2", "N3", "REM"]
STAGE_COLORS = {"WAKE": "tab:orange", "N1": "tab:cyan", "N2": "tab:blue", "N3": "darkblue", "REM": "tab:red"}

The usual start, now in one function: `load_epochs` (in `sleep_utils.py`) loads the night, filters 0.1–40 Hz, cuts it into 30-s epochs and finds the bad epochs (the detector based on Mariani et al., 2018 + movement time, as on Saturday). It returns the epochs, the stages, and the table of bad epochs (one row per epoch, the same columns as Saturday's file).

We keep the EEG in **µV** (`units="uV"`), so the power comes out in µV²/Hz.

In [ ]:
epochs, stages, artefacts = load_epochs(0, 2)  # SC4002, 33 years
onsets = epoch_onsets(epochs)
sf = epochs.info["sfreq"]  # 100 Hz

CHANNELS = ["Fpz-Cz", "Pz-Oz"]
eeg = epochs.get_data(picks=CHANNELS, units="uV")  # (n_epochs, 2 channels, 3000 samples)

bad = artefacts["bad"].to_numpy()  # True = bad epoch
clean = ~bad
print(f"{len(eeg)} epochs, {bad.sum()} bad ({bad.mean():.1%}), data shape {eeg.shape}")

## 1. The spectrum of one epoch

A **power spectrum** (power spectral density, PSD) tells how much of the signal's variance sits at each frequency. Unit: µV²/Hz. The area under the PSD between two frequencies is the power in that band (µV²).

We start with one N2 epoch with spindles (02:56:00, you marked it in session 03).

In [ ]:
i_n2 = np.flatnonzero(onsets.strftime("%H:%M:%S") == "02:56:00")[0]
x = eeg[i_n2, 0]  # Fpz-Cz
t = np.arange(len(x)) / sf

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(t, x, lw=0.6, color="black")
ax.set(xlabel="Time in the epoch (s)", ylabel="µV", title=f"Fpz-Cz, epoch {i_n2} ({stages[i_n2]}, 02:56:00)")
plt.show()

### Exercise 1.1: window length

**Welch's method** cuts the epoch into shorter windows (here with 50 % overlap), computes the spectrum of each window and averages them. The window length sets two things:
- the **frequency resolution**: the spacing between frequencies is Δf = 1 / window length
- the **noise**: more windows → more spectra averaged → a smoother estimate.

👉 Change `WINDOWS_S` (in seconds) and rerun: try `[30, 4]`, `[10, 4, 1]`, `[4, 2]`.

In [ ]:
WINDOWS_S = [30, 4]  # 👉 window lengths in seconds

fig, ax = plt.subplots(figsize=(6, 6))
for w in WINDOWS_S:
    f, p = welch(x, sf, nperseg=int(w * sf))  # 50 % overlap by default
    ax.plot(f, p, lw=1, label=f"{w} s windows: Δf = {f[1] - f[0]:.2f} Hz")
ax.set(xlim=(0, 40), ylim=(10e-5, 10e2,), yscale="log", xlabel="Frequency (Hz)", ylabel="PSD (µV²/Hz)", 
       title="One N2 epoch, Fpz-Cz")
ax.legend()
plt.show()

In [ ]:
WINDOW_S = 2  # the window we use from now on
# TODO: the frequency resolution of a 2-s window, in Hz
df = 1 / WINDOW_S
# TODO: how many 2-s windows with 50 % overlap fit in a 30-s epoch? (each new window starts 1 s after the previous one)
n_windows = int((30 - WINDOW_S) / (WINDOW_S / 2)) + 1
print(f"Δf = {df} Hz, {n_windows} windows averaged per epoch")

**Questions**
1. With 30-s windows (= one window, no averaging), what does the spectrum look like? Which window would you use to tell a 12-Hz from a 13-Hz spindle? Which to get a smooth spectrum?
2. The lowest frequency we care about is 0.5 Hz (delta). How many cycles of 0.5 Hz fit in a 4-s window? In a 1-s window?
3. Where is the **spindle** in the spectrum?

*Answers*
1. One 30-s window gives Δf = 0.033 Hz but a very noisy spectrum: each frequency is a single noisy estimate, so it jumps up and down by a factor of ~10. Our 2-s windows average 29 spectra: smooth, with Δf = 0.5 Hz, just enough to separate 12 from 13 Hz. 4-s windows (14 spectra, Δf = 0.25 Hz; used by the artefact detector and many sleep studies) are the other usual choice. There is no free lunch: resolution × number of averages is fixed by the epoch length.
2. 0.5 Hz has a period of 2 s, so a 4-s window holds 2 cycles (that's why many use 4 s, 2/0.5 Hz) and our 2-s window exactly 1: the lowest frequency it resolves is 0.5 Hz, right at the bottom of delta. A 1-s window holds half a cycle: the lowest frequency it can resolve is 1 Hz, so delta is badly estimated.
3. The bump around 12–14 Hz (the sigma band) above the smooth decrease of the spectrum.

### Exercise 1.2: linear or log?

The same spectrum, three ways.

In [ ]:
f, p = welch(x, sf, nperseg=int(WINDOW_S * sf))
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(f, p)
axes[0].set(title="Linear", ylabel="PSD (µV²/Hz)")
axes[1].semilogy(f, p)
axes[1].set(title="Log power")
axes[2].loglog(f[1:], p[1:])  # f[0] = 0 Hz can't go on a log axis
axes[2].set(title="Log power, log frequency")
for ax in axes:
    ax.set_xlabel("Frequency (Hz)")
plt.tight_layout()
plt.show()

**Questions**
1. On the linear plot, what can you see above 5 Hz?

*Answers*
1. Almost nothing: power falls by ~3 orders of magnitude from 1 to 30 Hz, so delta dominates the linear axis and the spindle bump disappears.

**Note**
On the log–log plot, the spectrum is almost a straight line with a few bumps. That line is the **aperiodic (1/f) component**; the bumps are **oscillations**. We come back to it in Section 6.

## 2. The spectrum of each stage

Now every epoch at once: `welch` works on the last axis, so it takes the whole `(n_epochs, 2, 3000)` array and returns `(n_epochs, 2, n_freqs)`. We keep 0–40 Hz (we low-passed at 40 Hz).

In [ ]:
freqs, psd = welch(eeg, sf, nperseg=int(WINDOW_S * sf))
keep = freqs <= 40
freqs, psd = freqs[keep], psd[:, :, keep]
# The same: psd[..., keep]. "..." stands for "all the axes before", so it also works on one spectrum (1 axis)
# or the stage spectra (2 axes). We use psd[..., keep] from now on.
print(psd.shape)

### Exercise 2.1: mean PSD per stage, clean epochs only

In [ ]:
stage_psd = {}
for stage in STAGES:
    # TODO: the mean PSD of the CLEAN epochs of this stage: shape (2, n_freqs)
    # Hint: (stages == stage) & clean is True for the epochs you want; psd[mask].mean(axis=0)
    stage_psd[stage] = psd[(stages == stage) & clean].mean(axis=0)

BAND_EDGES = [0.5, 4, 8, 12, 16, 30]  # delta | theta | alpha | sigma | beta | gamma


def plot_stage_psd(stage_psd, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    for k, (ax, ch) in enumerate(zip(axes, CHANNELS)):
        for stage, p in stage_psd.items():
            ax.semilogy(freqs, p[k], color=STAGE_COLORS[stage], label=stage, lw=1.5)
        for edge in BAND_EDGES:
            ax.axvline(edge, color="gray", lw=0.5, ls=":")
        ax.set(xlim=(0, 40), xlabel="Frequency (Hz)", title=f"{ch} {title}")
    axes[0].set_ylabel("PSD (µV²/Hz)")
    axes[1].legend()
    plt.show()


plot_stage_psd(stage_psd, "(clean epochs)")

**Questions** (the dotted lines are the band edges: delta 0.5–4, theta 4–8, alpha 8–12, sigma 12–16, beta 16–30, gamma 30–40 Hz)
1. Find the signature of each stage you know from the scoring rules (T2): alpha in relaxed wake, spindles in N2, slow waves in N3. On which channel is each one clearest?
2. Which two stages have the most similar spectra? Why is that a problem for automated staging with EEG only?
3. WAKE has more power than all sleep stages above ~20 Hz, on both channels, even on the clean epochs. Brain or muscle?
4. Where exactly is the alpha peak? Which band does it fall into with our band edges?

*Answers*
1. **Alpha** in WAKE, clearest on **Pz-Oz** (occipital, eyes closed). **Sigma** bump (~13 Hz) in N2 and N3, clearest on Fpz-Cz. **Delta**: N3 > N2 > the rest, by about an order of magnitude at 1 Hz between N3 and WAKE/REM; larger on Fpz-Cz (slow waves are frontal).
2. **REM and N1** (and to some extent REM and WAKE): low-amplitude mixed-frequency EEG, no spindles, no slow waves. The scorers separate them with the EOG (rapid eye movements) and the EMG (atonia), so a staging model needs those channels too.
3. Mostly muscle: in wake the scalp and neck muscles are tense, and EMG is broadband above ~20 Hz. The detector only removes epochs with *more* fast power than their neighbours (bursts), not the steady muscle tone of a whole wake period. So "clean" ≠ "free of muscle"; keep it in mind for any WAKE beta/gamma feature.
4. At ~8 Hz (7.5 Hz on Fpz-Cz, 8.5 Hz on Pz-Oz), right on the theta/alpha edge, so half of it is counted as "theta". Fixed bands assume everyone's alpha is at 8–12 Hz; it's slower in some people (and in old age). Alternative: bands relative to each person's **individual alpha frequency**. REM has a smaller peak at the same place (REM alpha).

## 3. Does artefact rejection matter?

The mean spectrum of each stage with all epochs (grey) and with the clean epochs only (colour).

In [ ]:
stage_psd_all = {stage: psd[stages == stage].mean(axis=0) for stage in STAGES}
CH = 0  # 👉 0 = Fpz-Cz, 1 = Pz-Oz

fig, axes = plt.subplots(1, len(STAGES), figsize=(16, 3.5), sharey=True)
for ax, stage in zip(axes, STAGES):
    n_bad = ((stages == stage) & bad).sum()
    ax.semilogy(freqs, stage_psd_all[stage][CH], color="gray", label="all epochs")
    ax.semilogy(freqs, stage_psd[stage][CH], color=STAGE_COLORS[stage], label="clean epochs")
    ax.set(xlim=(0, 40), xlabel="Frequency (Hz)", title=f"{stage}: {n_bad} of {(stages == stage).sum()} epochs bad")
    ax.legend()
axes[0].set_ylabel(f"PSD {CHANNELS[CH]} (µV²/Hz)")
plt.tight_layout()
plt.show()

**Questions**
1. Which stage changes most, and at which frequencies?
2. A few bad epochs change the **mean** PSD a lot. Why is the mean so sensitive? What would the **median** over epochs do?
3. For which feature would forgetting the artefact rejection be worst: N2 sigma power, or WAKE beta power?

*Answers*
1. **N1** and **WAKE**. On Fpz-Cz, N1's grey curve lies well above the clean one above ~20 Hz: only 5 of its 59 epochs are bad, but they raise the power at 30–40 Hz up to ~5× (muscle). WAKE (22 of 78 bad) is ~2× higher above 20 Hz (muscle) and below 2 Hz (movement). On Pz-Oz the difference is mostly below 2 Hz, in WAKE and N1. In N3, 9 bad epochs (of 297) still double the power above 25 Hz, because clean N3 has so little fast power; N2 and REM change little. The changes are where the detector looks (slow 0.5–4.5 Hz, fast 20–40 Hz), and small stages suffer most, because a few bad epochs weigh a lot in a short average.
2. Power is squared amplitude, so one 300-µV movement epoch weighs as much as dozens of normal epochs. The median (or the mean of log power) is much more robust, but it is not a fix: it hides artefacts rather than removing them, and the rest of the course uses clean epochs + mean.
3. WAKE (and N1) beta: that's exactly where muscle artefacts live (20–40 Hz). N2 sigma (12–16 Hz) is hardly touched.

## 4. Band power

A spectrum has ~160 numbers per epoch and channel. **Band power** summarises it in a few: the area under the PSD in each frequency band.

- **Absolute power** (µV²): the area in the band. Depends on everything that scales the signal: amplifier, electrode distance, skull thickness, age.
- **Relative power** (no unit, 0–1): the band's power divided by the total power (0.5–40 Hz). Removes the overall scaling, but now the bands depend on each other: if delta goes up, everything else goes down.

We use the lab's bands:

In [ ]:
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12), "sigma": (12, 16), "beta": (16, 30), "gamma": (30, 40)}
TOTAL = (0.5, 40)

### Exercise 4.1: a band-power function

The area under a curve: `simpson(y, x=x)` from `scipy.integrate` (Simpson's rule). With `axis=-1` it integrates along the last axis, so it works on one spectrum or on all epochs and channels at once.

In [ ]:
def band_power(freqs, psd, band):
    """Absolute power (µV²) in `band` = (low, high) Hz: the area under the PSD, along the last axis."""
    low, high = band
    # TODO: True for the frequencies from low to high (both included)
    in_band = (freqs >= low) & (freqs <= high)
    # TODO: the area under psd[..., in_band], with simpson(..., x=..., axis=-1)
    return simpson(psd[..., in_band], x=freqs[in_band], axis=-1)


# Band power of each stage spectrum (Section 2): stages x bands, Fpz-Cz, clean epochs (µV²)
stage_bp = pd.DataFrame({name: [band_power(freqs, stage_psd[stage][0], band) for stage in STAGES]
                         for name, band in BANDS.items()}, index=STAGES)
stage_bp.round(1)

YASA has the same function (`yasa.bandpower_from_psd_ndarray`). Do you get the same table?

In [ ]:
yasa_bands = [(lo, hi, name) for name, (lo, hi) in BANDS.items()]
yasa_bp = pd.DataFrame([yasa.bandpower_from_psd_ndarray(stage_psd[stage][0], freqs, bands=yasa_bands, relative=False)
                        for stage in STAGES], index=STAGES, columns=list(BANDS))
yasa_bp.round(1)

### Exercise 4.2: the band-power table

One row per epoch, one column per band, kind (abs/rel) and channel. This is the table the EEG features of the cohort are made from.

In [ ]:
bp = pd.DataFrame({"onset": onsets.strftime("%H:%M:%S"), "stage": stages, "bad": bad})
total = band_power(freqs, psd, TOTAL)  # (n_epochs, 2)
for name, band in BANDS.items():
    absolute = band_power(freqs, psd, band)  # (n_epochs, 2)
    # TODO: the relative power: absolute / total
    relative = absolute / total
    for k, ch in enumerate(["fpz", "pz"]):
        bp[f"{name}_abs_{ch}"] = absolute[:, k]
        bp[f"{name}_rel_{ch}"] = relative[:, k]

bp.to_csv(NIGHT_DIR / "bandpower_epochs.csv", index=False)
bp.head()

### Exercise 4.3: band power per stage

👉 `KIND` = `"rel"` or `"abs"`, `CH` = `"fpz"` or `"pz"`.

In [ ]:
KIND, CH = "rel", "fpz"  # 👉
cols = [f"{name}_{KIND}_{CH}" for name in BANDS]

# TODO: the median of each column, per stage, over the clean epochs only
# Hint: bp[~bp["bad"]].groupby("stage")[cols].median()
per_stage = bp[~bp["bad"]].groupby("stage")[cols].median().reindex(STAGES)
per_stage.columns = list(BANDS)

per_stage.plot.bar(stacked=(KIND == "rel"), figsize=(7, 4), colormap="viridis", rot=0,
                   ylabel="relative power" if KIND == "rel" else "µV²", title=f"Median band power, {CH}, {KIND}")
plt.legend(bbox_to_anchor=(1, 1))
plt.show()
per_stage.round(3 if KIND == "rel" else 1)

**Questions**
1. In relative power, which band dominates every stage? Does the stacked bar add up to exactly 1? Why not?
2. Switch to `"abs"`. How much larger is delta power in N3 than in N2, or in WAKE? Is the plot still readable?

*Answers*
1. Delta, everywhere (41 % in WAKE, 80 % in N2, 93 % in N3 on Fpz-Cz), because of the 1/f shape: the low frequencies always carry most of the power. The bars don't add up to 1 (WAKE: 0.92) because we took **medians**: the median of a sum is not the sum of the medians. The **means** add up to 1.00 (to within 0.5 %: the bins at 4, 8, 12, … Hz are in two bands, since both sides use `>=`/`<=`).
2. N3 delta is several times N2 delta, and ~10× WAKE (compare with the stage spectra). The other bands become invisible on a linear axis: same issue as in Exercise 1.2. A log axis (or log band power, which is what we will feed to the models) fixes it.

## 5. Band power across the night

Delta power in NREM sleep (**slow-wave activity**, SWA) is the classic marker of **sleep pressure** (Process S in Borbély's two-process model): it should be highest early in the night and decrease from cycle to cycle.

👉 Change `BAND`, `KIND` and `CH` and rerun. Grey = bad epochs (not shown), the hypnogram is below.

In [ ]:
BAND, KIND, CH = "delta", "abs", "fpz"  # 👉
SHOW = ["N2", "N3"]  # 👉 the stages to show, e.g. ["N2", "N3"] for SWA, ["REM"], ["WAKE"]

col = f"{BAND}_{KIND}_{CH}"
hours = (onsets - onsets[0]).total_seconds() / 3600

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for stage in SHOW:
    m = (stages == stage) & clean
    axes[0].scatter(hours[m], bp.loc[m, col], s=6, color=STAGE_COLORS[stage], label=stage)
axes[0].set(yscale="log" if KIND == "abs" else "linear", ylabel=col, title=f"{col} per clean epoch, SC4002")
axes[0].legend()

stage_y = {"WAKE": 4, "REM": 3, "N1": 2, "N2": 1, "N3": 0}
y = np.array([stage_y.get(s, np.nan) for s in stages])
axes[1].step(hours, y, where="post", color="black", lw=0.8)
axes[1].scatter(hours[bad], y[bad], color="gray", s=8, zorder=3, label="bad")
axes[1].set(yticks=list(stage_y.values()), yticklabels=list(stage_y.keys()), xlabel="Hours after lights-off")
axes[1].legend(loc="upper right")
plt.show()

### Exercise 5.1: SWA per cycle

A rough split of the night into cycles: hours 0–1.5, 1.5–3, 3–4.5, … (a real cycle detection is an optional exercise of Session 4).

In [ ]:
nrem_clean = np.isin(stages, ["N2", "N3"]) & clean
block = pd.cut(hours, bins=np.arange(0, hours.max() + 1.5, 1.5))  # the 1.5-h block of each epoch
# TODO: the median delta_abs_fpz of the clean N2+N3 epochs in each block. Hint, in three steps:
#   bp.loc[nrem_clean, "delta_abs_fpz"]          -> the values of the clean N2+N3 epochs
#   .groupby(block[nrem_clean], observed=True)    -> grouped by their block (observed=True: only blocks with epochs)
#   .median()                                     -> one median per block
swa = bp.loc[nrem_clean, "delta_abs_fpz"].groupby(block[nrem_clean], observed=True).median()
swa.round(0)

**Questions**
1. Does SWA decrease across the night? By how much from the first to the last block?
2. Part of the decrease is simply that there's less N3 late in the night. How would you show that SWA also decreases **within** a stage? Try `SHOW = ["N2"]`.
3. Try `BAND = "sigma"` in N2. Does sigma power follow the same course? (Spindles tend to increase across the night.)
4. Try `KIND = "rel"`. Relative delta follows the stage very closely. Why?

*Answers*
1. Overall yes (median ~1030 µV² in the first 1.5 h, ~180 µV² in the last), but **not monotonically** in this night: the 6–7.5 h block is back up at ~690 µV², because SC4002 has a long, deep N3 episode at ~6.5–7 h (look at the plot: as strong as the first cycle). The textbook decline is an average over many nights; one night is noisy, and fixed 1.5-h blocks don't follow the real cycles.
2. Look within one stage: N2 delta alone goes down steadily, ~340 → 250 → 180 µV² from the first to the last blocks. That's the homeostatic decay, not just less N3. (Within-cycle peaks of N3 are less clear here, see question 1.)
3. N2 sigma on Fpz-Cz goes the other way: ~7 µV² early, ~8–9 µV² in the second half of the night. Spindle activity tends to rise across the night while SWA falls.
4. Relative delta is mostly "how much of the signal is slow", which is what the scorers use to tell N2 from N3, so it follows the stage (0.79–0.92 in every block) and hardly shows the decline.

## 6. A young and an old subject

Last section: SC4002 (33 y) vs. SC4752 (96 y), the two nights you compared on Saturday. `night_spectra` (given) does Sections 2 and 3 for any night: `load_epochs`, and the mean PSD per stage over the clean epochs.

In [ ]:
def night_spectra(subject, night):
    """freqs, and the mean PSD (2 channels x n_freqs, µV²/Hz) of the clean epochs of each stage."""
    ep, st, art = load_epochs(subject, night)
    ok = ~art["bad"].to_numpy()
    f, p = welch(ep.get_data(picks=CHANNELS, units="uV"), sf, nperseg=int(WINDOW_S * sf))
    keep = f <= 40
    return f[keep], {stage: p[(st == stage) & ok][..., keep].mean(axis=0) for stage in STAGES if ((st == stage) & ok).any()}


NIGHTS = {"SC4002 (33 y)": (0, 2), "SC4752 (96 y)": (75, 2)}  # 👉 try other nights
spectra = {label: night_spectra(*sn)[1] for label, sn in NIGHTS.items()}

In [ ]:
STAGE, KIND = "N2", "abs"  # 👉 STAGE: "N2", "N3", "REM", "WAKE"; KIND: "abs" or "rel"

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for k, (ax, ch) in enumerate(zip(axes, CHANNELS)):
    for label, sp in spectra.items():
        if STAGE not in sp:
            continue
        p = sp[STAGE][k]
        if KIND == "rel":
            p = p / band_power(freqs, p, TOTAL)  # each spectrum divided by its total power
        ax.loglog(freqs[1:], p[1:], label=label)
    ax.set(xlim=(0.5, 40), xlabel="Frequency (Hz)", title=f"{ch}, {STAGE}")
axes[0].set_ylabel("PSD (µV²/Hz)" if KIND == "abs" else "PSD / total power (1/Hz)")
axes[1].legend()
plt.show()

### Exercise 6.1: the 1/f slope

On a log–log plot, the spectrum is roughly a straight line: power ∝ 1 / f^χ. The **exponent** χ (the negative slope) is the **aperiodic** part of the spectrum. A flatter spectrum (smaller χ) is often read as more "noise" or a shift in the balance of excitation and inhibition, and it gets flatter with age (Voytek et al., 2015).

Simplest estimate: a straight line through log10(power) vs. log10(frequency), with `np.polyfit(x, y, 1)`, which returns `[slope, intercept]`.

In [ ]:
def aperiodic_slope(freqs, p, fmin=2, fmax=30):
    """Slope of log10(power) vs. log10(frequency) between fmin and fmax (a straight-line fit)."""
    m = (freqs >= fmin) & (freqs <= fmax)
    # TODO: fit a line to log10(freqs[m]) and log10(p[m]); return the slope
    return np.polyfit(np.log10(freqs[m]), np.log10(p[m]), 1)[0]


slopes = pd.DataFrame({label: {f"{stage} {ch}": aperiodic_slope(freqs, sp[stage][k])
                               for stage in sp for k, ch in enumerate(CHANNELS)}
                       for label, sp in spectra.items()})
slopes.round(2)

**Questions**
1. In absolute power, the old subject has much less power, mostly at low frequencies. Is that "less slow-wave sleep", or could it be something else? Switch to `KIND = "rel"`: what is left of the difference?
2. Is the slope flatter in the old subject? In which stages?
3. A straight line through 2–30 Hz also goes through the sigma and alpha bumps. How does that bias the slope? (Exercise 6.3 fits the aperiodic part and the peaks separately.)
4. One young and one old night: what can you conclude? (On Wednesday you'll have every night.)

*Answers*
1. In N2 on Fpz-Cz, absolute delta power is halved (251 vs. 133 µV²), and the total power too (310 vs. 163 µV²); on Pz-Oz it's ~4× lower. But **relative** delta is the same on Fpz-Cz (0.81 vs. 0.81) and only a bit lower on Pz-Oz (0.77 vs. 0.72). So most of the N2 difference is a **scaling**, not a different shape. Slow waves do get smaller with age, but absolute power also drops for reasons unrelated to sleep: thicker skull and more CSF, electrode impedance, amplifier gain. In N3 a real shape difference remains (relative delta 0.93 vs. 0.86 on Fpz-Cz): the old subject's "N3" has fewer slow waves. That's why age effects on absolute power are hard to interpret, and why both kinds go into the feature table. (Below 0.5 Hz the old subject has *more* power on Fpz-Cz: drifts and eye movements, not brain.)
2. With the straight line: yes, in every stage and on both channels the old subject's spectrum is flatter (e.g. N3 Fpz-Cz: −3.05 vs. −2.33; N2: −2.57 vs. −2.07), as reported in the literature: the aperiodic exponent decreases with age. Exercise 6.3 shows that this doesn't hold in every stage once the peaks are taken out.
3. A peak in the fitted range lifts part of the line: a big sigma bump (young N2) pulls the fitted slope, a missing one doesn't. So part of the "slope" difference can be a difference in oscillations. That's what specparam separates (Exercise 6.3). Choosing a range without peaks (e.g. 30–45 Hz) isn't possible here: 100 Hz sampling and our 40-Hz low-pass.
4. Nothing about age yet: two people differ in many ways (Saturday's question). It's a hypothesis to test on the cohort.

### Exercise 6.2: the slope of every epoch

So far we fitted the slope to the **average** spectrum of a stage. The same fit works on every 30-s epoch, so we can follow the 1/f slope across the night, like the band power in Section 5. Back to SC4002 (`psd` from Section 2).

In [ ]:
for k, ch in enumerate(["fpz", "pz"]):
    # TODO: the slope (2-30 Hz) of every epoch on channel k, as a list: one aperiodic_slope(...) per epoch
    # Hint: psd[:, k] has one spectrum per epoch; [aperiodic_slope(freqs, p) for p in psd[:, k]]
    bp[f"slope_{ch}"] = [aperiodic_slope(freqs, p) for p in psd[:, k]]

bp.to_csv(NIGHT_DIR / "bandpower_epochs.csv", index=False)  # the epoch table now has the slopes too
bp[~bp["bad"]].groupby("stage")[["slope_fpz", "slope_pz"]].median().reindex(STAGES).round(2)  # median per stage

In [ ]:
CH = "fpz"  # 👉 "fpz" or "pz"
SHOW = STAGES  # 👉 the stages to show, e.g. ["N2", "N3"]

col = f"slope_{CH}"
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for stage in SHOW:
    m = (stages == stage) & clean
    axes[0].scatter(hours[m], bp.loc[m, col], s=6, color=STAGE_COLORS[stage], label=stage)
axes[0].set(ylabel=f"1/f slope, {CH} (2–30 Hz)", title=f"{col} per clean epoch, SC4002")
axes[0].legend(ncol=len(SHOW))

stage_y = {"WAKE": 4, "REM": 3, "N1": 2, "N2": 1, "N3": 0}  # the hypnogram, as in Section 5
y = np.array([stage_y.get(s, np.nan) for s in stages])
axes[1].step(hours, y, where="post", color="black", lw=0.8)
axes[1].scatter(hours[bad], y[bad], color="gray", s=8, zorder=3, label="bad")
axes[1].set(yticks=list(stage_y.values()), yticklabels=list(stage_y.keys()), xlabel="Hours after lights-off")
axes[1].legend(loc="upper right")
plt.show()

**Questions**
1. Does the slope follow the hypnogram? Which stages have the steepest and the flattest spectra? Could you score sleep from the slope alone?
2. Try `SHOW = ["N2"]`, then `["N3"]`. Does the slope change across the night within a stage? Compare with the slow-wave activity in Section 5.
3. In N2 and N3, the steepness goes together with the delta power. Is the per-epoch slope then an "aperiodic" measure, or mostly another way of measuring slow waves? (Exercise 6.3 helps.)

*Answers*
1. Yes, closely. Median slope on Fpz-Cz: N3 −3.05, N2 −2.51, N1 −1.99, REM −1.93, WAKE −1.67 (Pz-Oz the same order, a bit steeper). The deeper the NREM sleep, the steeper the spectrum. WAKE has the widest spread (muscle activity flattens it). REM and N1 overlap almost completely (10–90 % ranges −2.3 to −1.6 and −2.3 to −1.7), like their spectra in Section 2: the slope alone separates NREM depth well, but not REM from N1 or wake, for that you need the EOG and EMG.
2. Yes: within N2 the slope gets flatter across the night (median −2.74 in the first 1.5 h to −2.43 in the last), and within N3 too (−3.26 to −2.95), like the decline of SWA. The slope is used as a marker of sleep depth and of the "arousal level" of the cortex (e.g. Lendner et al., 2020, *eLife*), and its change across the night as a marker of sleep pressure.
3. Mostly slow waves, with a 2–30 Hz straight line: across NREM epochs the slope and log delta power are strongly correlated (Spearman ρ ≈ −0.9). A big slow-wave bump at 1–4 Hz lifts the low end of the spectrum and tilts the line. That's why specparam (Exercise 6.3) fits the peaks separately, and why some studies fit the slope only at high frequencies (e.g. 30–45 Hz), above the oscillations; not possible here with 100 Hz sampling and a 40-Hz low-pass.

### Exercise 6.3: separating the 1/f from the peaks with specparam

**specparam** (formerly FOOOF; Donoghue et al., 2020, *Nat Neurosci*) models the log power spectrum as an **aperiodic** part, offset − log10(f^χ) (a straight line on a log–log plot), plus **peaks** (Gaussians) on top of it, and fits both at once. So the exponent χ is no longer pulled by the spindle or alpha bumps.

Settings: a peak must be 1–8 Hz wide (`peak_width_limits`; with Δf = 0.5 Hz, narrower "peaks" are just noise), and at most 6 peaks.

In the plots: black = the spectrum, blue dashed = the aperiodic fit, red = the full model, green = the peaks. Both axes are log10.

👉 Change `STAGE` and `CH` and rerun this cell and the next one.

In [ ]:
STAGE, CH = "N2", 0  # 👉 STAGE: "N2", "N3", "REM", "WAKE"; CH: 0 = Fpz-Cz, 1 = Pz-Oz
FREQ_RANGE = [1, 35]  # Hz: the range we fit

models = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (label, sp) in zip(axes, spectra.items()):
    fm = SpectralModel(peak_width_limits=[1, 8], max_n_peaks=6, verbose=False)
    # TODO: fit the model to this night's spectrum of STAGE, channel CH, in FREQ_RANGE
    # Hint: fm.fit(freqs, power, FREQ_RANGE), where power is sp[STAGE][CH]
    fm.fit(freqs, sp[STAGE][CH], FREQ_RANGE)
    fm.plot(ax=ax, plt_log=True, plot_peaks="shade", add_legend=(ax is axes[0]))
    ax.set_title(f"{label}: {STAGE}, {CHANNELS[CH]}")
    models[label] = fm
plt.show()

In [ ]:
rows, peaks = {}, {}
for label, fm in models.items():
    # TODO: the aperiodic exponent of the model. Hint: fm.get_params("aperiodic") returns [offset, exponent]
    exponent = fm.get_params("aperiodic")[1]
    rows[label] = {"exponent (specparam)": exponent,
                   "-slope (straight line, 6.1)": -aperiodic_slope(freqs, spectra[label][STAGE][CH]),
                   "R² of the fit": fm.get_metrics("gof")}
    peaks[label] = pd.DataFrame(np.atleast_2d(fm.get_params("peak")), columns=["frequency (Hz)", "height", "width (Hz)"])

print(pd.DataFrame(rows).round(3), "\n")
pd.concat(peaks).round(1)  # the peaks of each night (height: in log10 power above the aperiodic fit)

**Questions**
1. Which peaks does specparam find in each night (green, and the table)? At what frequency is the spindle peak of each subject?
2. Compare the specparam exponent with the straight-line slope of Exercise 6.1. Which is larger? Is the difference between the young and the old subject still there, and is it as large?
3. 👉 Try `STAGE = "N3"`, `"REM"` and `"WAKE"`, and both channels. Is the old subject's exponent always smaller? Where does the model fit worst (R²), and why?
4. Try `STAGE = "N3"` with `FREQ_RANGE = [0.5, 35]`. Does a straight line on the log–log plot fit the whole range? What happens to the exponent?

*Answers*
1. N2, Fpz-Cz: the young subject has peaks at ~3, ~6 and **12.9 Hz**, the old one at ~6, ~11 and **13.9 Hz**. The spindle peak is the highest in both (height 0.7 vs. 0.5), and it is ~1 Hz faster in the old subject (Pz-Oz: 13.1 vs. 14.0 Hz); spindle frequency tends to increase slightly with age. The broad "peaks" at 3–7 Hz are bumps of delta/theta activity above the line: specparam calls anything above the aperiodic fit a peak, so read them with care.
2. The specparam exponent is **smaller** than the straight-line one (N2, Fpz-Cz: 2.18 vs. 2.57 young; 1.91 vs. 2.07 old): the delta bump at the low end and the spindle bump made the line steeper. The young–old difference is still there, but it shrinks from 0.50 to 0.27: part of the "flatter spectrum with age" in Exercise 6.1 was really *less delta and sigma activity* (fewer, smaller slow waves and spindles), not a different aperiodic part.
3. In **NREM**, yes, on both channels (N3: 2.77 vs. 2.12 on Fpz-Cz, 2.83 vs. 2.18 on Pz-Oz). In **REM** not: on Fpz-Cz the old subject's exponent is even larger (1.59 vs. 1.71). In **WAKE** the fits are the worst (R² 0.94–0.98 instead of > 0.99): the big alpha peak, and muscle activity that flattens the spectrum above ~20 Hz, which no straight line plus Gaussians describes well. So the stage matters: compare exponents within a stage (and NREM is where the age effect is clearest).
4. No: below ~1.5 Hz the N3 spectrum is flatter than above, so the line bends. A straight line over 0.5–35 Hz gives a smaller exponent (young: 2.31 instead of 2.77): the exponent depends on the range you fit, so report it and keep it fixed. A model with a bend ("knee") is the optional exercise.

## Optional exercise: the knee

On a log–log plot, the spectrum often **bends** at low frequencies: below a few Hz it is flatter than above. specparam's default (`aperiodic_mode="fixed"`) is a straight line. `aperiodic_mode="knee"` adds a bend: aperiodic = offset − log10(k + f^χ). Then `get_params("aperiodic")` returns [offset, k, exponent], and the bend (the **knee frequency**) is at k^(1/χ) Hz.

Fit both modes to the N3 spectrum (Fpz-Cz) of both nights between 0.5 and 35 Hz, plot the four fits, and compare the exponents, the knee frequencies and R². Then try N2, REM and Pz-Oz. Would you put the knee frequency into the cohort table?

In [ ]:
STAGE, CH = "N3", 0  # 👉
FREQ_RANGE_KNEE = [0.5, 35]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
rows = {}
for col, (label, sp) in enumerate(spectra.items()):
    for row, mode in enumerate(["fixed", "knee"]):
        fm = SpectralModel(aperiodic_mode=mode, peak_width_limits=[1, 8], max_n_peaks=6, verbose=False)
        fm.fit(freqs, sp[STAGE][CH], FREQ_RANGE_KNEE)
        ax = axes[row, col]
        fm.plot(ax=ax, plt_log=True, plot_peaks="shade", add_legend=(row == 0 and col == 0))
        ax.set_title(f"{label}: {mode}")
        ap = fm.get_params("aperiodic")  # fixed: [offset, exponent]; knee: [offset, k, exponent]
        knee_freq = ap[1] ** (1 / ap[2]) if mode == "knee" and ap[1] > 0 else np.nan  # k <= 0: no knee
        rows[(label, mode)] = {"exponent": ap[-1], "k": ap[1] if mode == "knee" else np.nan,
                               "knee frequency (Hz)": knee_freq, "R²": fm.get_metrics("gof")}
plt.tight_layout()
plt.show()
pd.DataFrame(rows).T.round(3)

*Answers (optional exercise)*

- **Young N3** (Fpz-Cz): the knee model fits better (R² 0.998 vs. 0.996), with a knee at ~1.4 Hz, and the exponent above the knee is much steeper (3.13 vs. 2.31 for the straight line over 0.5–35 Hz). On Pz-Oz: knee at ~1.3 Hz, R² 0.999 vs. 0.994.
- **Old N3**: hardly any bend (knee at ~0.7 Hz, the lowest bins only), the same R², and the exponent barely changes (2.21 vs. 2.03).
- **Other stages**: the knee is **unstable**. In some fits k comes out negative (old subject REM and WAKE on Fpz-Cz): there is no knee, and the knee frequency is undefined. In others it jumps (young REM Pz-Oz: 3 Hz; young WAKE Pz-Oz: 4 Hz) while R² gets *worse* than with the straight line.
- Why: with Δf = 0.5 Hz there are only 2–3 frequency bins below a 1–1.5 Hz knee, so it is poorly constrained. And below ~1 Hz the spectrum is also shaped by the recording: Sleep-EDF's hardware high-pass at 0.5 Hz and our 0.1-Hz filter, plus eye-movement drifts on Fpz-Cz. Part of the "knee" is the amplifier, not the brain.
- So no: not as a cohort feature from these data. The fixed exponent over a fixed range (e.g. 1–35 Hz, NREM) is more robust; the knee needs longer windows (finer Δf) and recordings without a 0.5-Hz hardware filter.

## What's next

`band_power` and the per-stage medians go into `process_night` in the **cohort notebook**, so the cohort table gets the EEG features (band power per stage, on the clean epochs only). Then we rerun all nights.